## Exercice n°1 (1/2 h)




Dans le modèle ci-dessous, pouvez-vous préciser :
- la taille du champ réceptif associé à un pixel d'une carte de caractéristique en sortie de enc3, par un calcul théorique.
- passer ce modèle sur un champ ne contenant que des zéros sauf pour une composante. En déduire la taille du champ réceptif empirique en sortie du modèle.


In [2]:
import torch
import torch.nn as nn

class MyNN(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()

        # 2 Conv2D
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_ch, base, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base, base, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # 2 Conv2D, first with stride=2
        self.enc2 = nn.Sequential(
            nn.Conv2d(base, base * 2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base * 2, base * 2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # again 2 Conv2D, first with stride=2
        self.enc3 = nn.Sequential(
            nn.Conv2d(base * 2, base * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base * 4, base * 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # alternance Conv2D / ConvTranspose2D
        self.mid = nn.Conv2d(base * 4, base * 4, kernel_size=3, padding=1, bias=False)

        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2, bias=False)
        self.dec1 = nn.Conv2d(base * 2, base * 2, kernel_size=3, padding=1, bias=False)

        self.up2 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2, bias=False)
        self.dec2 = nn.Conv2d(base, base, kernel_size=3, padding=1, bias=False)

        self.head = nn.Conv2d(base, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = torch.relu(self.mid(x))

        x = self.up1(x)
        x = torch.relu(self.dec1(x))

        x = self.up2(x)
        x = torch.relu(self.dec2(x))

        return self.head(x)

Prenons :

 k_i = taille du kernel de la couche i
 s_i = stride de la couche i
 p_i = padding de la couche i
 RF_i = champ réceptif après couche i
 j_i = jump (distance entre 2 pixels adjacents de la carte de caractéristiques)

On a la formule itérative :
          RF_i = RF_i-1 + (k_i - 1)*j_i
          
         et
         
          
          j_i = j_i-1*s_i
Soit
RF_0 = 1 (un pixel d’entrée)
j_0 = 1 (jump initial)


On a enc1 et enc2 :

* enc1 : 2 convs k=3, s=1, p=1
* enc2 : 1 conv k=3, s=2, p=1 + 1 conv k=3, s=1, p=1
* enc3 : conv k=3, s=2, p=1 + 1 conv k=3, s=1, p=1
self.enc3 = nn.Sequential(
    nn.Conv2d(base*2, base*4, kernel_size=3, stride=2, padding=1, bias=False),
    nn.ReLU(),
    nn.Conv2d(base*4, base*4, kernel_size=3, stride=1, padding=1, bias=False),
    nn.ReLU(),
)

# enc1
1. Première conv k=3, s=1, p=1
   
   RF = 1 + (3-1)*1 = 3, et j = 1*1 = 1
   

2. Deuxième conv k=3, s=1, p=1
   
   RF = 3 + (3-1)*1 = 5, et j = 1*1 = 1
   
#enc2

1. Première conv k=3, s=2, p=1
   
   RF = 5 + (3-1)*1 = 7,et j = 1*2 = 2
   

2. Deuxième conv k=3, s=1, p=1
   
   RF = 7 + (3-1)*2 = 7 + 4 = 11, et j = 2*1 = 2
   
#enc3

1. Première conv k=3, s=2, p=1
   
   RF = 11 + (3-1)*2 = 11 + 4 = 15,et  j = 2*2 = 4
   

2. Deuxième conv k=3, s=1, p=1
   
   RF = 15 + (3-1)*4 = 15 + 8 = 23, et j = 4*1 = 4
   
CONCLUSION : champ receptif théorique = 23 pixels

In [9]:
import torch

model = MyNN()
model.eval()

x = torch.randn(1, 3, 64, 64, requires_grad=True)

out = model.enc3(model.enc2(model.enc1(x)))

# pixel central
h, w = out.shape[2] // 2, out.shape[3] // 2

# somme sur les canaux pour être sûr d'avoir un gradient
out[0, :, h, w].sum().backward()

# gradient par pixel d'entrée
grad = x.grad.abs().sum(dim=1).squeeze()  # (H, W)

indices = grad.nonzero(as_tuple=False)

y_min, y_max = indices[:,0].min(), indices[:,0].max()
x_min, x_max = indices[:,1].min(), indices[:,1].max()

print("Champ réceptif empirique:",
      (y_max - y_min + 1, x_max - x_min + 1))


Champ réceptif empirique: (tensor(23), tensor(23))
